# Week 4 — VQC Ablation Study

**Goal:** Determine whether more VQC depth / more qubits closes the test AUC gap vs. classical.

| Experiment | Qubits | Layers | Expected |
|---|---|---|---|
| A1 `vqc_1layer` | 3 | 1 | Faster, may underfit |
| A2 `vqc_2layer_base` | 3 | 2 | Week 3 baseline (0.7812 test) |
| A3 `vqc_3layer` | 3 | 3 | Deeper — watch for barren plateau |
| A4 `vqc_2layer_5qubit` | 5 | 2 | More expressivity, less bottleneck |

All experiments: `lr=3e-5`, `dropout=0.4`, `weight_decay=1e-3`, `max_patches=3000`, 30 epochs max, early-stop patience=10.  
Estimated runtime: ~11–12 hours on RTX 5060 8 GB.

In [ ]:
# Cell 1 — Imports & setup
import os, sys, json, time, gc
from pathlib import Path

# Fix CUDA allocator fragmentation — set before any torch CUDA call
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:512'

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, f1_score
from torch_geometric.loader import DataLoader as PyGLoader

sys.path.insert(0, '..')
from pathq.model_v2 import QuantaPathV2
from pathq.dataset_v2 import get_loaders_from_features

FEATURES_DIR = Path('./data/features_uni')
CKPT_DIR     = Path('../checkpoints')
OUT_DIR      = Path('./outputs')
CKPT_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


In [ ]:
# Cell 2 — Load data (same 70/15/15 split as Week 3, seed=42)
print('Loading data...')
train_loader, val_loader, test_loader = get_loaders_from_features(
    FEATURES_DIR, batch_size=4, k=8, seed=42, num_workers=0, max_patches=3000
)
print(f'Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)} | Test: {len(test_loader.dataset)}')

# Quick label-balance check
tr_labels = [g.y.item() for g in train_loader.dataset]
va_labels = [g.y.item() for g in val_loader.dataset]
te_labels = [g.y.item() for g in test_loader.dataset]
print(f'Train pos/neg: {sum(tr_labels)}/{len(tr_labels)-sum(tr_labels)}')
print(f'Val   pos/neg: {sum(va_labels)}/{len(va_labels)-sum(va_labels)}')
print(f'Test  pos/neg: {sum(te_labels)}/{len(te_labels)-sum(te_labels)}')

In [ ]:
# Cell 3 — Training and evaluation functions

@torch.no_grad()
def eval_model(model, loader, device):
    model.eval()
    all_logits, all_labels = [], []
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        logits, _ = model(batch)
        loss = F.cross_entropy(logits, batch.y.view(-1))
        total_loss += loss.item()
        all_logits.append(logits.cpu())
        all_labels.append(batch.y.view(-1).cpu())
        del logits, loss
        if device.type == 'cuda':
            torch.cuda.empty_cache()
    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)
    probs     = torch.softmax(all_logits, dim=1)[:, 1].numpy()
    preds     = all_logits.argmax(dim=1).numpy()
    labels_np = all_labels.numpy()
    try:
        auc = roc_auc_score(labels_np, probs)
    except Exception:
        auc = 0.5
    f1  = f1_score(labels_np, preds, zero_division=0)
    tp  = ((preds==1)&(labels_np==1)).sum()
    tn  = ((preds==0)&(labels_np==0)).sum()
    fp  = ((preds==1)&(labels_np==0)).sum()
    fn  = ((preds==0)&(labels_np==1)).sum()
    return {
        'loss': total_loss / max(len(loader), 1),
        'auc':  auc, 'f1': f1,
        'sensitivity': tp / max(tp+fn, 1),
        'specificity': tn / max(tn+fp, 1),
    }


def run_ablation(cfg, train_loader, val_loader, test_loader, device, epochs=15):
    """Run one ablation experiment end-to-end. Auto-resumes from *_latest.pth."""
    label      = cfg['name']
    n_qubits   = cfg['n_qubits']
    vqc_layers = cfg['vqc_layers']
    lr         = cfg.get('lr', 3e-5)
    bs         = cfg.get('batch_size', 4)

    # Rebuild loaders if batch_size differs (5-qubit uses bs=2 for VRAM)
    if bs != 4:
        tr, va, te = get_loaders_from_features(
            FEATURES_DIR, batch_size=bs, k=8, seed=42, num_workers=0, max_patches=3000
        )
    else:
        tr, va, te = train_loader, val_loader, test_loader

    model = QuantaPathV2(
        use_vqc=True, n_qubits=n_qubits, vqc_layers=vqc_layers, dropout=0.4
    ).to(device)

    opt   = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-3
    )
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)

    ckpt_best   = CKPT_DIR / f'v4_{label}_best.pth'
    ckpt_latest = CKPT_DIR / f'v4_{label}_latest.pth'

    # --- Auto-resume ---
    best_auc    = 0.0
    start_epoch = 1
    history     = {'train_loss': [], 'val_loss': [], 'val_auc': [], 'val_f1': []}
    if ckpt_latest.exists():
        saved = torch.load(ckpt_latest, map_location=device, weights_only=False)
        model.load_state_dict(saved['model_state'])
        opt.load_state_dict(saved['opt_state'])
        sched.load_state_dict(saved['sched_state'])
        best_auc    = saved['best_auc']
        start_epoch = saved['epoch'] + 1
        history     = saved.get('history', history)
        print(f'[{label}] Resumed from epoch {saved["epoch"]}, best_auc={best_auc:.4f}')

    print(f'\n{"="*72}')
    print(f'[{label}]  {n_qubits}q x {vqc_layers}L  lr={lr}  batch={bs}  epochs={epochs}')
    print(f'{"="*72}')
    print(f'{"Ep":>4}  {"TrLoss":>8}  {"VaLoss":>8}  {"AUC":>7}  {"F1":>7}  '
          f'{"Sens":>7}  {"Spec":>7}  {"Time":>7}  Best')
    print('-'*72)

    patience     = 0
    early_stop_n = 10

    for epoch in range(start_epoch, epochs + 1):
        t0 = time.time()
        model.train()
        tr_loss = 0.0
        n = 0
        for batch in tr:
            batch = batch.to(device)
            opt.zero_grad()
            logits, _ = model(batch)
            loss_val = loss_fn = F.cross_entropy(logits, batch.y.view(-1))
            loss_item = loss_val.item()          # extract .item() BEFORE backward
            loss_fn.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            del logits, loss_val, loss_fn        # release compute graph immediately
            tr_loss += loss_item
            n += 1
            if device.type == 'cuda':
                torch.cuda.empty_cache()
        tr_loss /= max(n, 1)

        vm = eval_model(model, va, device)
        sched.step()

        is_best = vm['auc'] > best_auc
        if is_best:
            best_auc = vm['auc']
            patience = 0
            torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                        'best_auc': best_auc, 'config': cfg}, ckpt_best)
        else:
            patience += 1

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vm['loss'])
        history['val_auc'].append(vm['auc'])
        history['val_f1'].append(vm['f1'])

        torch.save({
            'epoch': epoch, 'model_state': model.state_dict(),
            'opt_state': opt.state_dict(), 'sched_state': sched.state_dict(),
            'best_auc': best_auc, 'config': cfg, 'history': history,
        }, ckpt_latest)

        elapsed = time.time() - t0
        overfit_flag = '  WARNING' if vm['loss'] > tr_loss * 2.5 else ''
        best_flag    = ' NEW BEST' if is_best else overfit_flag
        print(f'{epoch:>4}  {tr_loss:>8.4f}  {vm["loss"]:>8.4f}  {vm["auc"]:>7.4f}  '
              f'{vm["f1"]:>7.4f}  {vm["sensitivity"]:>7.4f}  {vm["specificity"]:>7.4f}  '
              f'{elapsed:>5.0f}s{best_flag}')

        if patience >= early_stop_n:
            print(f'Early stop — no improvement for {early_stop_n} epochs')
            break

    # --- Final evaluation on best checkpoint ---
    saved = torch.load(ckpt_best, map_location=device, weights_only=False)
    model.load_state_dict(saved['model_state'])
    test_m = eval_model(model, te, device)
    val_m  = eval_model(model, va, device)
    gap = val_m['auc'] - test_m['auc']

    print(f'\n[{label}] FINAL  val={val_m["auc"]:.4f}  '
          f'test={test_m["auc"]:.4f}  gap={gap:.4f} '
          f'({"healthy" if gap < 0.06 else "OVERFIT"})')

    del model
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()

    return {
        'name':              label,
        'n_qubits':          n_qubits,
        'vqc_layers':        vqc_layers,
        'val_auc':           val_m['auc'],
        'test_auc':          test_m['auc'],
        'test_f1':           test_m['f1'],
        'test_sensitivity':  test_m['sensitivity'],
        'test_specificity':  test_m['specificity'],
        'val_test_gap':      gap,
        'history':           history,
    }

print('Functions defined.')


In [ ]:
# Cell 4 — Sanity check: 1 forward pass per config, no training
print('Sanity-checking all 4 configs...')

configs = [
    {'name': 'vqc_1layer',        'n_qubits': 3, 'vqc_layers': 1, 'lr': 3e-5, 'batch_size': 4},
    {'name': 'vqc_2layer_base',   'n_qubits': 3, 'vqc_layers': 2, 'lr': 3e-5, 'batch_size': 4},
    {'name': 'vqc_3layer',        'n_qubits': 3, 'vqc_layers': 3, 'lr': 3e-5, 'batch_size': 4},
    {'name': 'vqc_2layer_5qubit', 'n_qubits': 5, 'vqc_layers': 2, 'lr': 3e-5, 'batch_size': 2},
]

sample_batch = next(iter(train_loader)).to(device)

for cfg in configs:
    m = QuantaPathV2(
        use_vqc=True, n_qubits=cfg['n_qubits'], vqc_layers=cfg['vqc_layers'], dropout=0.4
    ).to(device)
    with torch.no_grad():
        logits, _ = m(sample_batch)
    n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'  [{cfg["name"]:24s}]  logits={logits.shape}  params={n_params:,}  OK')
    del m
    if device.type == 'cuda':
        torch.cuda.empty_cache()

print('\nAll 4 configs pass. Ready to run ablation.')

In [ ]:
# Cell 5 — Run all 4 ablation experiments
# Runtime: ~15 epochs x ~108 min/epoch x 4 configs = ~108 hours worst case
# Early stop (patience=10) will cut this significantly
# If interrupted: re-run this cell — auto-resumes from v4_<name>_latest.pth

results = []

for cfg in configs:
    res = run_ablation(
        cfg, train_loader, val_loader, test_loader, device, epochs=15
    )
    results.append(res)

    # Save intermediate JSON after each completed experiment
    out = {r['name']: {k: v for k, v in r.items() if k != 'history'} for r in results}
    out_path = OUT_DIR / 'week4_ablation_results.json'
    with open(out_path, 'w') as f:
        json.dump(out, f, indent=2)
    print(f'[{len(results)}/4] Saved to {out_path}\n')

print('='*72)
print('ALL 4 ABLATION EXPERIMENTS COMPLETE')
print('='*72)


In [ ]:
# Cell 6 — Results table
SEP = '-' * 82
print(SEP)
print(f'{"Config":<24}  {"nQ":>3}  {"L":>2}  {"ValAUC":>7}  {"TestAUC":>8}  '
      f'{"F1":>6}  {"Sens":>6}  {"Spec":>6}  {"Gap":>7}')
print(SEP)

for r in results:
    flag = 'OK' if r['val_test_gap'] < 0.06 else 'OVERFIT'
    print(f'{r["name"]:<24}  {r["n_qubits"]:>3}  {r["vqc_layers"]:>2}  '
          f'{r["val_auc"]:>7.4f}  {r["test_auc"]:>8.4f}  '
          f'{r["test_f1"]:>6.4f}  {r["test_sensitivity"]:>6.4f}  '
          f'{r["test_specificity"]:>6.4f}  {r["val_test_gap"]:>7.4f}  {flag}')

print(SEP)
print(f'[W3 ref] vqc_2layer        3   2   0.8217   0.7812  0.4828  0.4375  0.8235   0.0405  OK')
print(SEP)

best_test = max(results, key=lambda r: r['test_auc'])
best_gap  = min(results, key=lambda r: r['val_test_gap'])
print(f'\nHighest test AUC    -> {best_test["name"]} ({best_test["test_auc"]:.4f})')
print(f'Best generalisation -> {best_gap["name"]}  (gap={best_gap["val_test_gap"]:.4f})')

In [ ]:
# Cell 7 — 6-panel figure
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Week 4 — VQC Ablation Study (QuantaPath v2, CAMELYON16)',
             fontsize=13, fontweight='bold')

COLORS      = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
SHORT_NAMES = ['1L/3Q', '2L/3Q\n(base)', '3L/3Q', '2L/5Q']
W3_COLOR    = 'gray'


def bar_labels(ax, bars, vals):
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.005,
                f'{v:.4f}', ha='center', va='bottom', fontsize=8)


# Panel 1: Training loss curves
ax = axes[0, 0]
for i, r in enumerate(results):
    ax.plot(r['history']['train_loss'], color=COLORS[i], label=SHORT_NAMES[i])
ax.set_title('Training Loss'); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Panel 2: Validation AUC curves
ax = axes[0, 1]
for i, r in enumerate(results):
    ax.plot(r['history']['val_auc'], color=COLORS[i], label=SHORT_NAMES[i])
ax.axhline(0.8217, color=W3_COLOR, linestyle='--', alpha=0.7, label='W3 ref')
ax.set_title('Validation AUC'); ax.set_xlabel('Epoch'); ax.set_ylabel('AUC')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Panel 3: Test AUC bar chart
ax = axes[0, 2]
test_aucs = [r['test_auc'] for r in results]
bars = ax.bar(SHORT_NAMES, test_aucs, color=COLORS, alpha=0.85, edgecolor='black')
ax.axhline(0.7812, color=W3_COLOR, linestyle='--', alpha=0.7, label='W3 ref')
ax.set_title('Test AUC'); ax.set_ylabel('AUC'); ax.set_ylim(0.5, 1.0)
bar_labels(ax, bars, test_aucs)
ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis='y')

# Panel 4: Val->Test gap
ax = axes[1, 0]
gaps = [r['val_test_gap'] for r in results]
bars = ax.bar(SHORT_NAMES, gaps, color=COLORS, alpha=0.85, edgecolor='black')
ax.axhline(0.0405, color=W3_COLOR, linestyle='--', alpha=0.7, label='W3 ref')
ax.set_title('Val->Test Gap (lower = better)')
ax.set_ylabel('Gap'); ax.set_ylim(0, max(gaps + [0.15]))
bar_labels(ax, bars, gaps)
ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis='y')

# Panel 5: Test F1
ax = axes[1, 1]
f1s  = [r['test_f1'] for r in results]
bars = ax.bar(SHORT_NAMES, f1s, color=COLORS, alpha=0.85, edgecolor='black')
ax.axhline(0.4828, color=W3_COLOR, linestyle='--', alpha=0.7, label='W3 ref')
ax.set_title('Test F1'); ax.set_ylabel('F1'); ax.set_ylim(0, 1.0)
bar_labels(ax, bars, f1s)
ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis='y')

# Panel 6: Sensitivity vs Specificity
ax = axes[1, 2]
for i, r in enumerate(results):
    ax.scatter(r['test_specificity'], r['test_sensitivity'],
               color=COLORS[i], label=SHORT_NAMES[i].replace('\n', ''), s=160, zorder=5)
    ax.annotate(SHORT_NAMES[i].replace('\n', ' '),
                (r['test_specificity'] + 0.01, r['test_sensitivity'] - 0.02),
                fontsize=7)
ax.scatter(0.8235, 0.4375, color=W3_COLOR, marker='x', s=160, label='W3 ref', zorder=5)
ax.set_title('Sensitivity vs Specificity')
ax.set_xlabel('Specificity'); ax.set_ylabel('Sensitivity')
ax.set_xlim(0, 1.1); ax.set_ylim(0, 1.1)
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = OUT_DIR / 'week4_ablation_results.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved -> {fig_path}')

In [ ]:
# Cell 8 — Select best config, paper narrative, save JSONs

best_by_auc = max(results, key=lambda r: r['test_auc'])
best_by_gen = min(results, key=lambda r: r['val_test_gap'])

print('=' * 60)
print('WEEK 4 ABLATION SUMMARY')
print('=' * 60)

print('\nRanked by test AUC:')
for r in sorted(results, key=lambda x: x['test_auc'], reverse=True):
    flag = 'OK' if r['val_test_gap'] < 0.06 else 'OVERFIT'
    print(f'  {r["name"]:<24}  AUC={r["test_auc"]:.4f}  F1={r["test_f1"]:.4f}  '
          f'gap={r["val_test_gap"]:.4f} [{flag}]')

print(f'\n* Best test AUC:      {best_by_auc["name"]} ({best_by_auc["test_auc"]:.4f})')
print(f'* Best generalisation: {best_by_gen["name"]} (gap={best_by_gen["val_test_gap"]:.4f})')

print('\n--- Paper Narrative ---')
print('Week 3 baseline (2L/3Q): test_auc=0.7812, gap=0.0405')
for r in results:
    delta = r['test_auc'] - 0.7812
    trend = f'+{delta:.4f}' if delta >= 0 else f'{delta:.4f}'
    print(f'  {r["name"]:<24}  delta_test_AUC={trend}')

best_config = {
    'week': 4,
    'best_by_test_auc': {
        'name':         best_by_auc['name'],
        'n_qubits':     best_by_auc['n_qubits'],
        'vqc_layers':   best_by_auc['vqc_layers'],
        'test_auc':     best_by_auc['test_auc'],
        'val_test_gap': best_by_auc['val_test_gap'],
        'checkpoint':   f'checkpoints/v4_{best_by_auc["name"]}_best.pth',
    },
    'best_by_generalisation': {
        'name':         best_by_gen['name'],
        'n_qubits':     best_by_gen['n_qubits'],
        'vqc_layers':   best_by_gen['vqc_layers'],
        'test_auc':     best_by_gen['test_auc'],
        'val_test_gap': best_by_gen['val_test_gap'],
        'checkpoint':   f'checkpoints/v4_{best_by_gen["name"]}_best.pth',
    },
    'all_results': [{k: v for k, v in r.items() if k != 'history'} for r in results],
    'week3_reference': {
        'test_auc': 0.7812, 'val_auc': 0.8217, 'val_test_gap': 0.0405,
        'test_f1': 0.4828, 'n_qubits': 3, 'vqc_layers': 2,
    },
}

cfg_path = OUT_DIR / 'week4_best_config.json'
with open(cfg_path, 'w') as f:
    json.dump(best_config, f, indent=2)
print(f'\nSaved -> {cfg_path}')
print('Week 4 complete.')